<a href="https://colab.research.google.com/github/yusrpro9/radar/blob/main/notebooks/datasets_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Datasets Preparation

This notebook prepares the external [RAID](https://huggingface.co/datasets/liamdugan/raid) dataset for use alongside the PAN-2026 training data.

**Goals:**
- Filter RAID to genres and attack types relevant to the PAN-2026 task.
- Create stratified dev / test splits and upload them to the HuggingFace Hub as `username/raid-dataset`.
- Save both splits locally for use in the preprocessing and training notebooks.

**Key decisions:**
- Genres retained: `news`, `books`, `poetry` — domains closest to PAN-2026 (essays, fiction, news).
- Attack types retained: `none`, `homoglyph`, `synonym`, `paraphrase`, `alternative_spelling` — attacks that mirror the adversarial obfuscations present in PAN-2026 test data.
- Split ratio: 60 % dev / 40 % test, stratified by label.

## Setup

In [ ]:
%%writefile requirements.txt
datasets==4.0.0
huggingface_hub==1.11.0
pandas==2.2.2
tqdm==4.67.3
numpy==2.0.2

Writing requirements.txt


In [ ]:
!pip install -r /content/requirements.txt --quiet

## Imports

In [ ]:
from tqdm.auto import tqdm
from pathlib import Path
import pandas as pd
import datasets
from datasets import load_dataset, ClassLabel

## Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
HF_USERNAME = userdata.get('HF_USERNAME')
login(token=HF_TOKEN)

In [ ]:
from pathlib import Path

class CFG:
    PROJECT_DIR: Path = Path("/content/drive/MyDrive/robust-ai-text-detector")
    DATA_DIR: Path = PROJECT_DIR / "data"
    HF_RAID_REPO: str = f"{HF_USERNAME}/raid-dataset"
    RANDOM_SEED: int = 42


## PAN-2026 Dataset

Load the official PAN-2026 training and validation splits from the HuggingFace Hub.

In [ ]:
pan26_ds = load_dataset(f"{HF_USERNAME}/pan26")
train_df = pan26_ds['train'].to_pandas()
val_df = pan26_ds['validation'].to_pandas()

README.md:   0%|          | 0.00/529 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/56.7M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/8.42M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/23707 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3589 [00:00<?, ? examples/s]

**Training split** — 23,707 samples across 5 columns: `id`, `text`, `model`, `label`, `genre`.

In [ ]:
train_df.head()

,id,text,model,label,genre
0,ea468d03-1973-5039-86b2-ff225bb92c4e,"Duke Ellington, a titan of jazz, revolutionize...",falcon3-10b-instruct,1,essays
1,0d05f269-6d67-521d-9b5d-cc18f482c6c1,I reflected on the shifting dynamics of media ...,o3-mini,1,essays
2,c2ec79f3-da80-58f8-bef0-3e0ea7ab072f,"In F. Scott Fitzgerald's ""The Great Gatsby,"" t...",gpt-4o,1,essays
3,4ad37c58-0bb7-536b-997d-cfccabd0d094,I still chuckle when I think about that time I...,deepseek-r1-distill-qwen-32b,1,essays
4,07747b0c-5051-5e0d-8096-b4d4ed8bd98e,"Yoga, originating in ancient India, encompasse...",gemini-2.0-flash,1,essays


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23707 entries, 0 to 23706
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      23707 non-null  object
 1   text    23707 non-null  object
 2   model   23707 non-null  object
 3   label   23707 non-null  int64 
 4   genre   23707 non-null  object
dtypes: int64(1), object(4)
memory usage: 926.2+ KB


**Validation split** — 3,589 samples with the same schema.

In [ ]:
val_df.head()

,id,text,model,label,genre
0,7caf42b9-fd48-5e97-a0d0-0ae28a1f9603,"In William Faulkner's ""The Sound and the Fury,...",gpt-4o,1,essays
1,28b61fc4-e82b-5cf8-bc34-1ecdb7182993,"Manipulation, a profound and pervasive theme i...",gpt-4.5-preview,1,essays
2,22398c76-da72-5724-973e-0981b8e9cbee,Edna's journey is a testament to her rebellion...,llama-3.3-70b-instruct,1,essays
3,3cd1e50d-e1f0-5f8f-bfb8-0b8a6048bcaa,There are three main aspects of the gun contro...,human,0,essays
4,6e5745a6-0335-50cc-bdf0-fa0e1fee7518,During the Portuguese colonial period in Angol...,gpt-4o,1,essays


In [ ]:
val_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3589 entries, 0 to 3588
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      3589 non-null   object
 1   text    3589 non-null   object
 2   model   3589 non-null   object
 3   label   3589 non-null   int64 
 4   genre   3589 non-null   object
dtypes: int64(1), object(4)
memory usage: 140.3+ KB


## RAID Dataset

The RAID benchmark ([Dugan et al., 2024](https://arxiv.org/abs/2406.07441)) contains 5.6 M rows generated by 11 LLMs across 11 domains and 12 adversarial attack types.

Column schema after loading: `id`, `adv_source_id`, `source_id`, `model`, `decoding`, `repetition_penalty`, `attack`, `domain`, `title`, `prompt`, `generation`.

### Loading and Column Selection

In [ ]:
raid_raw = load_dataset("liamdugan/raid", "raid", split="train")

README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/11.8G [00:00<?, ?B/s]

extra.csv:   0%|          | 0.00/3.71G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating extra split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

In [ ]:
raid_raw.column_names

['id',
 'adv_source_id',
 'source_id',
 'model',
 'decoding',
 'repetition_penalty',
 'attack',
 'domain',
 'title',
 'prompt',
 'generation']

Retain only the columns required for training; rename `generation` → `text` and `domain` → `genre` to align with the PAN-2026 schema.

In [ ]:
raid_ds = raid_raw.select_columns(["id", "adv_source_id", "source_id", "generation", "model", "attack", "domain"])
raid_ds = raid_ds.rename_columns({"generation": "text", "domain": "genre"})
raid_ds.column_names

['id', 'adv_source_id', 'source_id', 'text', 'model', 'attack', 'genre']

Add integer `label` column: `0` = human, `1` = machine, mirroring the PAN-2026 convention.

In [ ]:
labels = [0 if model == "human" else 1 for model in raid_ds["model"]]
raid_ds = raid_ds.add_column(name="label", column=labels)

In [ ]:
raid_ds

Dataset({
    features: ['id', 'adv_source_id', 'source_id', 'text', 'model', 'attack', 'genre', 'label'],
    num_rows: 5615820
})

### Genre and Attack Filtering

**Genre filter** — keep `news`, `books`, and `poetry`.
These three domains are the closest semantic equivalents of PAN-2026's `news`, `fiction`, and `essays` genres, and they provide sufficient volume (≈2.2 M rows) after filtering.

In [ ]:
GENRES = ["news", "books", "poetry"]
raid_genre = raid_ds.filter(lambda x: x["genre"] in GENRES)

Filter:   0%|          | 0/5615820 [00:00<?, ? examples/s]

In [ ]:
raid_genre

Dataset({
    features: ['id', 'adv_source_id', 'source_id', 'text', 'model', 'attack', 'genre', 'label'],
    num_rows: 2239440
})

**Attack filter** — keep only the five attack types listed below.

| Attack | Rationale |
|--------|-----------|
| `none` | Clean AI/human baseline samples |
| `homoglyph` | Present in PAN-2026 adversarial test; character-level substitution |
| `synonym` | Lexical paraphrase that changes vocabulary but preserves structure |
| `paraphrase` | Semantic-level rewrite — hardest attack type |
| `alternative_spelling` | British/American spelling variants |

Excluded: `number`, `article_deletion`, `insert_paragraphs`, `perplexity_misspelling`, `upper_lower`, `whitespace`, `zero_width_space` — these introduce low-level noise not representative of PAN-2026 obfuscations.

In [ ]:
ATTACKS = ["none", "homoglyph", "synonym", "paraphrase", "alternative_spelling"]
raid_filtered = raid_genre.filter(lambda x: x["attack"] in ATTACKS)

Filter:   0%|          | 0/2239440 [00:00<?, ? examples/s]

In [ ]:
raid_filtered

Dataset({
    features: ['id', 'adv_source_id', 'source_id', 'text', 'model', 'attack', 'genre', 'label'],
    num_rows: 933100
})

Cast the `label` column to `ClassLabel` to enable stratified splitting.

In [ ]:
class_labels = ClassLabel(names=[0, 1], num_classes=2)
raid_filtered = raid_filtered.cast_column("label", class_labels)

Casting the dataset:   0%|          | 0/933100 [00:00<?, ? examples/s]

### Splitting and Saving

Split 60 % → `dev` (supplementary training pool) / 40 % → `test` (external evaluation), stratified by label.

| Split | Rows | Purpose |
|-------|------|---------|
| dev | 559,860 | External training pool (used to augment PAN-2026 train) |
| test | 373,240 | External held-out evaluation (out-of-domain robustness) |

In [ ]:
raid_split = raid_filtered.train_test_split(
    test_size=0.4,
    stratify_by_column="label",
    seed=CFG.RANDOM_SEED,
)

In [ ]:
raid_split

DatasetDict({
    train: Dataset({
        features: ['id', 'adv_source_id', 'source_id', 'text', 'model', 'attack', 'genre', 'label'],
        num_rows: 559860
    })
    test: Dataset({
        features: ['id', 'adv_source_id', 'source_id', 'text', 'model', 'attack', 'genre', 'label'],
        num_rows: 373240
    })
})

Upload both splits to the HuggingFace Hub for reproducibility and use in downstream notebooks.

In [ ]:
raid_split['train'].push_to_hub(CFG.HF_RAID_REPO, split="dev")
raid_split['test'].push_to_hub(CFG.HF_RAID_REPO, split="test")

Save both splits locally for offline use in Colab.

In [ ]:
(CFG.DATA_DIR / "raid").mkdir(parents=True, exist_ok=True)
raid_split['train'].to_pandas().to_json(
    CFG.DATA_DIR / "raid" / "dev.jsonl", orient='records', lines=True
)
raid_split['test'].to_pandas().to_json(
    CFG.DATA_DIR / "raid" / "test.jsonl", orient='records', lines=True
)

In [ ]:
(CFG.DATA_DIR / "pan26").mkdir(parents=True, exist_ok=True)
pan26_ds['train'].to_pandas().to_json(
    CFG.DATA_DIR / "pan26" / "train.jsonl", orient='records', lines=True
)
pan26_ds['validation'].to_pandas().to_json(
    CFG.DATA_DIR / "pan26" / "val.jsonl", orient='records', lines=True
)

Quick sanity-check — verify saved files load correctly.

In [ ]:
pd.read_json(CFG.DATA_DIR / "raid" / "dev.jsonl", lines=True).head()

,id,adv_source_id,source_id,text,model,attack,genre,label
0,db2a9624-9532-463b-b331-0ae86b52091b,db2a9624-9532-463b-b331-0ae86b52091b,b47ec949-0c37-4320-8f13-bcfdeb0f466e,The Conservatives have attacked plans to allow...,mistral,none,news,1
1,39035d74-7b5b-4e35-b272-0753b12f89db,39035d74-7b5b-4e35-b272-0753b12f89db,047fd9c7-bd71-4595-9111-bd5c4b6a4030,"In the small town of Ashwood, a sense of uneas...",llama-chat,none,books,1
2,cd35f3cd-ff98-4cfe-b7f1-12abbd5baeb7,cd35f3cd-ff98-4cfe-b7f1-12abbd5baeb7,083a11a3-11e8-4cf1-b680-2b282012cfb4,"In one corner, we have what most Internet user...",mistral,none,news,1
3,cd81e6d2-f5ac-4044-b189-917c37658b4d,b8fe902f-6b80-48b6-9db4-370fad5a6afd,ac92dd15-0327-4f71-a41c-2356f0d03d08,"Α summеr аftеrnооn, wаrm аnd brіght,\nFіllеd w...",mistral-chat,homoglyph,poetry,1
4,fee3f9ab-2073-4b7e-87f1-fdc5c167b3f8,5fc8431e-c9b1-4591-90a4-de69f093566e,cb00a512-d69f-4739-a4bd-35e98689feda,The novel is “The Sweet Sorrow of the Virtuous...,mistral,paraphrase,books,1


In [ ]:
pd.read_json(CFG.DATA_DIR / "raid" / "test.jsonl", lines=True).head()

,id,adv_source_id,source_id,text,model,attack,genre,label
0,724c9a00-60c1-4c1d-abed-829d4f3c472c,66e9923e-35ae-4d06-b08c-5877057becb4,c507e6bd-e8a3-4c3b-9655-6c293ef4cdd9,"""Winter of Fire"" is a novel by John Grisham. I...",gpt2,alternative_spelling,books,1
1,b7951422-5015-4313-9d5a-28fdb9993623,b7951422-5015-4313-9d5a-28fdb9993623,fe252deb-2d6b-4ed6-bac4-315e1dc40e94,In a world where magic and science coexist in ...,llama-chat,none,books,1
2,8ac46e63-4d4b-4fb9-9b74-a2eb6a8e7f53,b1c21d2d-c8a2-4b5c-9eaf-745aec675192,5bc30d4a-edae-4345-8ecc-aa2ae3c6465f,"A family in New York, consisting mostly of Ita...",mpt,paraphrase,books,1
3,09a970d7-7d30-42f7-9db3-6486e301bf45,363416c0-1d93-466a-91ba-7967aaf31fef,127290d6-516e-45b0-91bd-a580e79d31d6,"Come dawn, come dusk, it's still so unclear\nT...",cohere-chat,alternative_spelling,poetry,1
4,d4685589-5846-414e-9061-19b6aef339ba,77807ada-c737-49d2-98e4-404a660d5529,7eaa19d9-dced-4d29-bf3e-8f64261380b9,Sроnsоrеd Αds\n\nТhе fіrst еvеr Wіzаrd Оf Οz s...,mistral,homoglyph,news,1


In [ ]:
pd.read_json(CFG.DATA_DIR / "pan26" / "train.jsonl", lines=True).head()

,id,text,model,label,genre
0,ea468d03-1973-5039-86b2-ff225bb92c4e,"Duke Ellington, a titan of jazz, revolutionize...",falcon3-10b-instruct,1,essays
1,0d05f269-6d67-521d-9b5d-cc18f482c6c1,I reflected on the shifting dynamics of media ...,o3-mini,1,essays
2,c2ec79f3-da80-58f8-bef0-3e0ea7ab072f,"In F. Scott Fitzgerald's ""The Great Gatsby,"" t...",gpt-4o,1,essays
3,4ad37c58-0bb7-536b-997d-cfccabd0d094,I still chuckle when I think about that time I...,deepseek-r1-distill-qwen-32b,1,essays
4,07747b0c-5051-5e0d-8096-b4d4ed8bd98e,"Yoga, originating in ancient India, encompasse...",gemini-2.0-flash,1,essays


In [ ]:
pd.read_json(CFG.DATA_DIR / "pan26" / "val.jsonl", lines=True).head()

,id,text,model,label,genre
0,7caf42b9-fd48-5e97-a0d0-0ae28a1f9603,"In William Faulkner's ""The Sound and the Fury,...",gpt-4o,1,essays
1,28b61fc4-e82b-5cf8-bc34-1ecdb7182993,"Manipulation, a profound and pervasive theme i...",gpt-4.5-preview,1,essays
2,22398c76-da72-5724-973e-0981b8e9cbee,Edna's journey is a testament to her rebellion...,llama-3.3-70b-instruct,1,essays
3,3cd1e50d-e1f0-5f8f-bfb8-0b8a6048bcaa,There are three main aspects of the gun contro...,human,0,essays
4,6e5745a6-0335-50cc-bdf0-fa0e1fee7518,During the Portuguese colonial period in Angol...,gpt-4o,1,essays


## Summary

### What this notebook does

1. **Loads PAN-2026** (`username/pan26`) — 23,707 training samples and 3,589 validation samples across three genres (fiction, essays, news) and 22 AI models.  
2. **Loads and filters RAID** (`liamdugan/raid`) — starting from 5,615,820 rows, filtering to:
   - Genres: `news`, `books`, `poetry` → **2,239,440 rows**
   - Attacks: `none`, `homoglyph`, `synonym`, `paraphrase`, `alternative_spelling` → **933,100 rows**
3. **Splits RAID** 60/40 stratified by label:
   - `dev` split: **559,860 rows** — supplementary training pool
   - `test` split: **373,240 rows** — external out-of-domain evaluation
4. **Uploads** both RAID splits to `username/raid-dataset` on the HuggingFace Hub.
5. **Saves** both splits locally to `data/raid/{dev,test}.jsonl`.

### Why these choices

| Decision | Rationale |
|----------|-----------|
| Genres: news / books / poetry | Closest semantic equivalents to PAN-2026's news / fiction / essays; avoids web, code, and recipe domains that introduce domain shift |
| Attacks: 5 of 11 retained | Focus on attacks that mirror real-world obfuscations likely to appear in ELOQUENT's PAN-2026 test set; numeric/whitespace attacks add noise without adversarial value |
| 60/40 split | Large enough `dev` pool for balanced augmentation; 40 % `test` gives statistically reliable out-of-domain evaluation |
| Stratified split | Label distribution in RAID is heavily skewed toward AI texts; stratification ensures both splits have identical human/machine ratios |

### Key statistics

| Dataset | Total | Human | Machine |
|---------|-------|-------|---------|
| PAN-2026 train | 23,707 | 9,101 (38.4 %) | 14,606 (61.6 %) |
| PAN-2026 val | 3,589 | ~1,277 (35.6 %) | ~2,312 (64.4 %) |
| RAID dev | 559,860 | stratified | stratified |
| RAID test | 373,240 | stratified | stratified |

### Next steps

The resulting files are consumed by `notebooks/preprocessing.ipynb`, which performs EDA, merges the sources, and produces the final balanced `train / val / test` splits used by the training notebook (`notebooks/RADAR.ipynb`).